In [1]:
# Bibliotecas utilizadas
import pandas as pd
import import_ipynb

In [2]:
from sales_analysis.utils.loggers.logger import *

In [3]:
def get_quality_test(df: pd.DataFrame) -> None:
    """
    Executa verificações básicas de qualidade de dados em um DataFrame.

    A função realiza validações relacionadas à:
    - Estrutura dos dados (shape)
    - Valores ausentes
    - Registros duplicados
    - Valores negativos
    - Qualidade de datas
    - Regras de negócio financeiras

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contendo os dados a serem avaliados.

    Returns
    -------
    None
        Exibe os resultados diretamente no console.

    Notes
    -----
    Regras de negócio avaliadas:
    - Produtos vendidos abaixo do custo unitário
    - Produtos vendidos abaixo do custo após desconto

    Examples
    --------
    >>> get_quality_test(df_sales)
    🔍 CHECKLIST DE QUALIDADE DE DADOS
    📌 Shape: (1000, 8)
    ⚠️ Valores Nulos: 5
    """

    logger.info(f"Início do teste de qualidade")
    
    print("🔍 CHECKLIST DE QUALIDADE DE DADOS\n")
    
    # Padronização
    df = df.replace(['', ' ', 'NA', 'N/A', 'null', None, '-'], pd.NA)

    # Informações gerais
    print("📌 Shape:", df.shape)

    # Nulos
    nulos = df.isna().sum().sum()

    if nulos > 0:
        print("⚠️ Valores Nulos:", nulos)
    else:
        print("📌 Valores Nulos:", nulos)

    # Duplicados
    duplicados = df.duplicated().sum()

    if duplicados > 0:
        print("⚠️ Valores Duplicados:", duplicados)
    else:
        print("📌 Valores Duplicados:", duplicados)

    # Negativos
    negativos = (df.select_dtypes(include="number") < 0).sum().sum()

    if negativos > 0:
        print("⚠️ Valores Negativos:", negativos)
    else:
        print("📌 Valores Negativos:", negativos)


    # Datas
    if "Sale_Date" in df.columns:

        # Tipo da data
        if not pd.api.types.is_datetime64_any_dtype(df["Sale_Date"]):
            print("⚠️ Tipagem da data:", df["Sale_Date"].dtype)
        else:
            print("📌 Tipagem da data:", df["Sale_Date"].dtype)

        # Datas nulas
        print("📌 Datas nulas:", df["Sale_Date"].isna().sum())

        # Datas mín e máx
        print("📌 Data mínima:", df["Sale_Date"].min())
        print("📌 Data máxima:", df["Sale_Date"].max())

    # Regras de negócio
    colunas_financeiras = ["Unit_Price", "Unit_Cost", "Discount"]

    if all(col in df.columns for col in colunas_financeiras):
        faturamento_sem_desc = (df["Unit_Price"] < df["Unit_Cost"]).sum()

        faturamento_com_desc = ((df["Unit_Price"] * (1 - df["Discount"])) < df["Unit_Cost"]).sum()

        total = len(df)

        if faturamento_sem_desc > 0:
            print(
                f"⚠️ Faturamento sem desconto (inconsistências): "
                f"{faturamento_sem_desc} ({faturamento_sem_desc / total:.2%})")
        else:
            print(
                f"📌 Faturamento sem desconto (inconsistências): "
                f"{faturamento_sem_desc} ({faturamento_sem_desc / total:.2%})")

        if faturamento_com_desc > 0:
            print(
                f"⚠️ Faturamento com desconto (inconsistências): "
                f"{faturamento_com_desc} ({faturamento_com_desc / total:.2%})")
        else:
            print(
                f"📌 Faturamento com desconto (inconsistências): "
                f"{faturamento_com_desc} ({faturamento_com_desc / total:.2%})")
            
    logger.info(f"Término do teste de qualidade")